In [ ]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Chargement et préparation des données
data = pd.read_csv("SOCR-HeightWeight.csv")
data.rename(columns={"Height(Inches)": "Height", "Weight(Pounds)": "Weight"}, inplace=True)
data.head()

In [ ]:
# Exploration rapide du dataset
print("Dimensions :", data.shape)
print("\nStatistiques descriptives :")
print(data[["Height", "Weight"]].describe().round(2).T)
print("\nValeurs manquantes :", data[["Height", "Weight"]].isnull().sum().to_dict())
print(f"\nCorrélation Height-Weight : {data['Height'].corr(data['Weight']):.4f}")

In [ ]:
# Visualisation des données brutes
plt.figure(figsize=(6, 5))
sns.scatterplot(x=data["Height"], y=data["Weight"], alpha=0.7, s=10)
plt.xlabel("Taille (pouces)")
plt.ylabel("Poids (livres)")
plt.title("Taille vs Poids")
plt.tight_layout()
plt.show()

In [ ]:
# Séparation train / test (80% / 20%)
from sklearn.model_selection import train_test_split

x = data["Height"].values
y = data["Weight"].values

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(f"Train : {len(x_train)} observations")
print(f"Test  : {len(x_test)} observations")

In [ ]:
# Normalisation (standardisation z-score) sur le train
# IMPORTANT : on calcule mean/std sur le train uniquement,
# puis on les applique au test pour éviter toute fuite de données.

x_mean, x_std = x_train.mean(), x_train.std()
y_mean, y_std = y_train.mean(), y_train.std()

x_train_n = (x_train - x_mean) / x_std
y_train_n = (y_train - y_mean) / y_std

x_test_n  = (x_test  - x_mean) / x_std
y_test_n  = (y_test  - y_mean) / y_std

print(f"x_train normalisé — mean: {x_train_n.mean():.4f}, std: {x_train_n.std():.4f}")

In [ ]:
# Fonction de coût (MSE / 2)
def cost_function(x, y, b, w):
    m = len(x)
    predictions = w * x + b
    total_cost = np.sum((predictions - y) ** 2)
    return total_cost / (2 * m)


# Descente de gradient (une itération)
def gradient_descent_step(x, y, b, w, learning_rate):
    m = len(x)
    predictions = w * x + b
    residuals  = predictions - y
    dw = (1 / m) * np.sum(residuals * x)
    db = (1 / m) * np.sum(residuals)
    w -= learning_rate * dw
    b -= learning_rate * db
    return b, w

In [ ]:
# Initialisation des hyperparamètres
b = 0.0
w = 0.0
learning_rate  = 0.01   # adapté aux données normalisées
num_iterations = 10_000
tol = 1e-9              # tolérance stricte pour une vraie convergence

cost_history = []

In [ ]:
# Entraînement du modèle
converged_at = None

for i in range(num_iterations):
    b, w = gradient_descent_step(x_train_n, y_train_n, b, w, learning_rate)
    cost = cost_function(x_train_n, y_train_n, b, w)
    cost_history.append(cost)

    # Vérification de convergence (différence relative entre deux itérations)
    if i > 0 and abs(cost_history[-2] - cost) < tol:
        converged_at = i
        print(f"Convergence atteinte à l'itération {i}")
        break

if converged_at is None:
    print(f"Entraînement terminé après {num_iterations} itérations.")

print(f"Paramètres normalisés — b: {b:.6f}, w: {w:.6f}")

In [ ]:
# Dénormalisation : retrouver b et w dans l'espace d'origine
#
# En espace normalisé :  y_n = w * x_n + b
# En espace original  :  y   = w_orig * x + b_orig
#
# w_orig = w * (y_std / x_std)
# b_orig = y_mean + y_std * b - w_orig * x_mean

w_orig = w * (y_std / x_std)
b_orig = y_mean + y_std * b - w_orig * x_mean

print(f"Paramètres finaux (espace original) :")
print(f"  w = {w_orig:.4f}  (gain de poids par pouce de taille)")
print(f"  b = {b_orig:.4f}  (ordonnée à l'origine)")

In [ ]:
# Courbe d'apprentissage (évolution du coût)
plt.figure(figsize=(7, 4))
plt.plot(cost_history, color="steelblue", linewidth=1.5)
plt.xlabel("Itération")
plt.ylabel("Coût (MSE / 2, espace normalisé)")
plt.title("Courbe d'apprentissage — descente de gradient")
plt.tight_layout()
plt.show()

In [ ]:
# Évaluation du modèle
def evaluate(x, y, b, w, label=""):
    preds = w * x + b
    residuals = y - preds
    mse  = np.mean(residuals ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(residuals))
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    print(f"--- {label} ---")
    print(f"  R²   : {r2:.4f}")
    print(f"  RMSE : {rmse:.4f} lbs")
    print(f"  MAE  : {mae:.4f} lbs")
    return r2, rmse, mae

print("Résultats en espace original :\n")
r2_train, rmse_train, _ = evaluate(x_train, y_train, b_orig, w_orig, "Train")
print()
r2_test,  rmse_test,  _ = evaluate(x_test,  y_test,  b_orig, w_orig, "Test")

In [ ]:
# Visualisation finale : nuage de points + droite de régression
x_line = np.linspace(x.min(), x.max(), 200)
y_line = w_orig * x_line + b_orig

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Graphique 1 : données brutes + droite ---
axes[0].scatter(x_test, y_test, alpha=0.2, s=8, label="Test set")
axes[0].plot(x_line, y_line, color="crimson", linewidth=2,
             label=f"Régression : y = {w_orig:.2f}x + ({b_orig:.2f})")
axes[0].set_xlabel("Taille (pouces)")
axes[0].set_ylabel("Poids (livres)")
axes[0].set_title("Taille vs Poids — droite de régression")
axes[0].legend(fontsize=9)

# --- Graphique 2 : résidus ---
preds_test = w_orig * x_test + b_orig
residuals  = y_test - preds_test

axes[1].scatter(preds_test, residuals, alpha=0.2, s=8, color="steelblue")
axes[1].axhline(0, color="crimson", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("Valeurs prédites (lbs)")
axes[1].set_ylabel("Résidus (lbs)")
axes[1].set_title("Analyse des résidus")

plt.suptitle(f"Modèle final — R² test = {r2_test:.4f} | RMSE test = {rmse_test:.2f} lbs",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Prédiction avec validation de la plage
#
# La régression linéaire simple est fiable uniquement dans les limites
# des données d'entraînement. En dehors, le modèle extrapole sans garantie.

HEIGHT_MIN = float(x_train.min())
HEIGHT_MAX = float(x_train.max())

def predict(height_in_inches):
    """
    Prédit le poids (en livres) à partir de la taille (en pouces).

    Paramètres
    ----------
    height_in_inches : float
        Taille en pouces. Doit être dans la plage des données d'entraînement.

    Retourne
    --------
    float : poids prédit en livres.

    Lève
    ----
    TypeError  : si la valeur n'est pas numérique.
    ValueError : si la valeur est hors de la plage d'entraînement.
    """
    if not isinstance(height_in_inches, (int, float)):
        raise TypeError(
            f"La taille doit être un nombre (reçu : {type(height_in_inches).__name__})"
        )
    if height_in_inches < HEIGHT_MIN or height_in_inches > HEIGHT_MAX:
        raise ValueError(
            f"Valeur hors plage : {height_in_inches:.1f} in — "
            f"le modèle accepte entre {HEIGHT_MIN:.1f} et {HEIGHT_MAX:.1f} in."
        )
    return w_orig * height_in_inches + b_orig


# --- Tests ---
print(f"Plage valide : {HEIGHT_MIN:.2f} – {HEIGHT_MAX:.2f} pouces\n")

# Cas valides
for h in [62.0, 67.5, 72.0]:
    print(f"  predict({h}) = {predict(h):.2f} lbs  ✓")

print()

# Cas invalides
for h in [150.0, 50.0, 'soixante']:
    try:
        result = predict(h)
        print(f"  predict({h}) = {result:.2f}  [DEVRAIT ÉCHOUER]")
    except (ValueError, TypeError) as e:
        print(f"  predict({h!r}) → {type(e).__name__}: {e}  ✓")
